# Task 3: AQE Experiment (Queries 2, 4)

Compare the same Gold-table queries with Adaptive Query Execution (AQE) disabled and enabled.

## Connect to Spark

In [1]:
import sys

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import execute_registered_query, register_integrated_taxi_trips

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-AQE-Queries3-5')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/19 10:24:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Configure the experiment

Only `spark.sql.adaptive.enabled` changes between the two runs. AQE uses runtime statistics to re-optimize a plan.

In [ ]:
QUERY_NAMES = (
    'query_02_bj',
    'query_04',
)
REPETITIONS = 3

AQE_DISABLED = 'false'
AQE_ENABLED = 'true'

## Measure AQE disabled and enabled

Each mode receives one warm-up followed by three timed runs.

In [5]:
def measure_aqe_mode(enabled):
    spark.catalog.clearCache()
    spark.conf.set('spark.sql.adaptive.enabled', enabled)
    register_integrated_taxi_trips(spark)
    return {
        query_name: measure_registered_query(spark, query_name, REPETITIONS)
        for query_name in QUERY_NAMES
    }

aqe_disabled_results = measure_aqe_mode(AQE_DISABLED)
aqe_enabled_results = measure_aqe_mode(AQE_ENABLED)

for query_name in QUERY_NAMES:
    assert rows_match(
        aqe_disabled_results[query_name]['rows'],
        aqe_enabled_results[query_name]['rows'],
    ), f'AQE results differ for {query_name}.'

print('AQE-enabled and AQE-disabled results match within float tolerance.')

26/09/19 10:31:17 WARN HintErrorLogger: Count not find relation 'air_quality' specified in hint 'BROADCAST(air_quality)'.
26/09/19 10:31:19 WARN HintErrorLogger: Count not find relation 'air_quality' specified in hint 'BROADCAST(air_quality)'.
26/09/19 10:31:21 WARN HintErrorLogger: Count not find relation 'air_quality' specified in hint 'BROADCAST(air_quality)'.
26/09/19 10:31:22 WARN HintErrorLogger: Count not find relation 'air_quality' specified in hint 'BROADCAST(air_quality)'.
26/09/19 10:31:24 WARN HintErrorLogger: Count not find relation 'air_quality' specified in hint 'BROADCAST(air_quality)'.
26/09/19 10:31:25 WARN HintErrorLogger: Count not find relation 'air_quality' specified in hint 'BROADCAST(air_quality)'.
26/09/19 10:31:27 WARN HintErrorLogger: Count not find relation 'air_quality' specified in hint 'BROADCAST(air_quality)'.
26/09/19 10:31:28 WARN HintErrorLogger: Count not find relation 'air_quality' specified in hint 'BROADCAST(air_quality)'.


AQE-enabled and AQE-disabled results match within float tolerance.


## AQE-disabled physical plans

In [6]:
spark.conf.set('spark.sql.adaptive.enabled', AQE_DISABLED)
register_integrated_taxi_trips(spark)

for query_name in QUERY_NAMES:
    print(f'\n--- {query_name} ---')
    execute_registered_query(spark, query_name).explain('formatted')


--- query_02_bj ---


26/09/19 10:31:35 WARN HintErrorLogger: Count not find relation 'air_quality' specified in hint 'BROADCAST(air_quality)'.


== Physical Plan ==
* Sort (15)
+- Exchange (14)
   +- * HashAggregate (13)
      +- Exchange (12)
         +- * HashAggregate (11)
            +- * Project (10)
               +- * BroadcastHashJoin Inner BuildRight (9)
                  :- * Project (4)
                  :  +- * Filter (3)
                  :     +- * ColumnarToRow (2)
                  :        +- Scan parquet  (1)
                  +- BroadcastExchange (8)
                     +- * Filter (7)
                        +- * ColumnarToRow (6)
                           +- Scan parquet  (5)


(1) Scan parquet 
Output [3]: [trip_distance#4982, coco_pu#5021, date#5036]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
PushedFilters: [IsNotNull(coco_pu)]
ReadSchema: struct<trip_distance:double,coco_pu:int>

(2) ColumnarToRow [codegen id : 2]
Input [3]: [trip_distance#4982, coco_pu#5021, date#5036]

(3) Filter [codegen id : 2]
Input [3]: [trip_distance#4982, coco_pu#5021, date

## AQE-enabled physical plans

AEQ enabled -> `AdaptiveSparkPlan` and changed shuffles or joins.

In [7]:
spark.conf.set('spark.sql.adaptive.enabled', AQE_ENABLED)
register_integrated_taxi_trips(spark)

for query_name in QUERY_NAMES:
    print(f'\n--- {query_name} ---')
    query = execute_registered_query(spark, query_name)
    query.collect()
    query.explain('formatted')


--- query_02_bj ---


26/09/19 10:31:38 WARN HintErrorLogger: Count not find relation 'air_quality' specified in hint 'BROADCAST(air_quality)'.


== Physical Plan ==
AdaptiveSparkPlan (33)
+- == Final Plan ==
   ResultQueryStage (21)
   +- * Sort (20)
      +- AQEShuffleRead (19)
         +- ShuffleQueryStage (18), Statistics(sizeInBytes=496.0 B, rowCount=14)
            +- Exchange (17)
               +- * HashAggregate (16)
                  +- AQEShuffleRead (15)
                     +- ShuffleQueryStage (14), Statistics(sizeInBytes=2.8 KiB, rowCount=67)
                        +- Exchange (13)
                           +- * HashAggregate (12)
                              +- * Project (11)
                                 +- * BroadcastHashJoin Inner BuildRight (10)
                                    :- * Project (4)
                                    :  +- * Filter (3)
                                    :     +- * ColumnarToRow (2)
                                    :        +- Scan parquet  (1)
                                    +- BroadcastQueryStage (9), Statistics(sizeInBytes=1024.2 KiB, rowCount=27)
             

## Benchmark summary

In [8]:
summary = []
for query_name in QUERY_NAMES:
    disabled = aqe_disabled_results[query_name]
    enabled = aqe_enabled_results[query_name]
    summary.append({
        'query': query_name,
        'aqe_disabled_samples_seconds': [
            round(value, 3) for value in disabled['samples_seconds']
        ],
        'aqe_disabled_median_seconds': round(disabled['median_seconds'], 3),
        'aqe_enabled_samples_seconds': [
            round(value, 3) for value in enabled['samples_seconds']
        ],
        'aqe_enabled_median_seconds': round(enabled['median_seconds'], 3),
        'aqe_speedup_factor': round(
            disabled['median_seconds'] / enabled['median_seconds'],
            2,
        ),
    })
summary

[{'query': 'query_02_bj',
  'aqe_disabled_samples_seconds': [1.654, 1.553, 1.456],
  'aqe_disabled_median_seconds': 1.553,
  'aqe_enabled_samples_seconds': [1.483, 1.494, 1.37],
  'aqe_enabled_median_seconds': 1.483,
  'aqe_speedup_factor': 1.05}]

## Cleanup

In [9]:
spark.catalog.clearCache()
spark.stop()